In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------
# 1. Настройки
# ------------------------------
IMG_SHAPE = (28, 28, 1)
LATENT_DIM = 20  # размер латентного пространства
BATCH_SIZE = 128
EPOCHS = 10

# ------------------------------
# 2. Загрузка MNIST
# ------------------------------
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# ------------------------------
# 3. Sampling слой для VAE
# ------------------------------
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps

# ------------------------------
# 4. Encoder
# ------------------------------
encoder_inputs = keras.Input(shape=IMG_SHAPE)
x = layers.Conv2D(32, 3, activation='relu', padding='same', strides=2)(encoder_inputs)
x = layers.Conv2D(64, 3, activation='relu', padding='same', strides=2)(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation='relu')(x)
z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

# ------------------------------
# 5. Decoder
# ------------------------------
latent_inputs = keras.Input(shape=(LATENT_DIM,))
x = layers.Dense(7*7*64, activation='relu')(latent_inputs)
x = layers.Reshape((7,7,64))(x)
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

# ------------------------------
# 6. VAE модель с кастомным loss
# ------------------------------
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        x, _ = data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x, training=True)
            reconstruction = self.decoder(z, training=True)

            # MSE loss
            rec_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - reconstruction), axis=[1,2,3]))

            # KL divergence
            kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1))

            loss = rec_loss + kl_loss

        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"loss": loss, "rec_loss": rec_loss, "kl_loss": kl_loss}

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())

# ------------------------------
# 7. Обучение
# ------------------------------
vae.fit(x_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(x_test, None))

# ------------------------------
# 8. Визуализация латентного пространства
# ------------------------------
z_mean, _, _ = encoder.predict(x_test)
plt.figure(figsize=(6,6))
plt.scatter(z_mean[:,0], z_mean[:,1], s=2, alpha=0.5)
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.title("2D latent space")
plt.show()

# ------------------------------
# 9. Генерация новых образов
# ------------------------------
n = 10
figure = np.zeros((28*n, 28*n))
grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)
for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])
        x_decoded = decoder.predict(z_sample)
        digit = x_decoded[0].reshape(28,28)
        figure[i*28:(i+1)*28, j*28:(j+1)*28] = digit
plt.figure(figsize=(8,8))
plt.imshow(figure, cmap='gray')
plt.axis('off')
plt.show()